In [1]:
import pandas as pd

df_train = pd.read_csv('../data/original/train.csv')
df_test = pd.read_csv('../data/original/test.csv')
df_complete = pd.read_csv('../data/original/titanic3.csv')

In [2]:
# Find names in `df_test` that contain ", which is not properly formatted in the source.
df_test[df_test['Name'].str.contains("\"", na=False)]

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
19,911,3,"Assaf Khalil, Mrs. Mariana (Miriam"")""",female,45.00,0,0,2696,7.2250,NaN,C
33,925,3,"Johnston, Mrs. Andrew G (Elizabeth Lily"" Watson)""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
35,927,3,"Katavelas, Mr. Vassilios (Catavelas Vassilios"")""",male,18.50,0,0,2682,7.2292,NaN,C
49,941,3,"Coutts, Mrs. William (Winnie Minnie"" Treanor)""",female,36.00,0,2,C.A. 37671,15.9000,NaN,S
52,944,2,"Hocking, Miss. Ellen Nellie""""",female,20.00,2,1,29105,23.0000,NaN,S
104,996,3,"Thomas, Mrs. Alexander (Thamine Thelma"")""",female,16.00,1,1,2625,8.5167,NaN,C
108,1000,3,"Willer, Mr. Aaron (Abi Weller"")""",male,NaN,0,0,3410,8.7125,NaN,S
144,1036,1,"Lindeberg-Lind, Mr. Erik Gustaf (Mr Edward Lin...",male,42.00,0,0,17475,26.5500,NaN,S
225,1117,3,"Moubarek, Mrs. George (Omine Amenia"" Alexander)""",female,NaN,0,2,2661,15.2458,NaN,C
244,1136,3,"Johnston, Master. William Arthur Willie""""",male,NaN,1,2,W./C. 6607,23.4500,NaN,S


In [3]:
# Find names in df_complete that contain "
df_complete[df_complete['name'].str.contains("\"", na=False)]

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
13,1,1,"Barber, Miss. Ellen ""Nellie""",female,26.0,0,0,19877,78.8500,NaN,S,6,NaN,NaN
37,1,1,"Bradley, Mr. George (""George Arthur Brayton"")",male,NaN,0,0,111427,26.5500,NaN,S,9,NaN,"Los Angeles, CA"
99,1,1,"Duff Gordon, Lady. (Lucille Christiana Sutherl...",female,48.0,1,0,11755,39.6000,A16,C,1,NaN,London / Paris
100,1,1,"Duff Gordon, Sir. Cosmo Edmund (""Mr Morgan"")",male,49.0,1,0,PC 17485,56.9292,A20,C,1,NaN,London / Paris
109,1,1,"Flynn, Mr. John Irwin (""Irving"")",male,36.0,0,0,PC 17474,26.3875,E25,S,5,NaN,"Brooklyn, NY"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1149,3,1,"Riordan, Miss. Johanna ""Hannah""",female,NaN,0,0,334915,7.7208,NaN,Q,13,NaN,NaN
1174,3,0,"Sage, Miss. Dorothy Edith ""Dolly""",female,NaN,8,2,CA. 2343,69.5500,NaN,S,NaN,NaN,NaN
1244,3,1,"Thomas, Mrs. Alexander (Thamine ""Thelma"")",female,16.0,1,1,2625,8.5167,NaN,C,14,NaN,NaN
1291,3,0,"Willer, Mr. Aaron (""Abi Weller"")",male,NaN,0,0,3410,8.7125,NaN,S,NaN,NaN,NaN


In [4]:
# Remove " from names containing " from both dataframes
df_test['Name'] = df_test['Name'].str.replace('"', '', regex=False)
df_complete['name'] = df_complete['name'].str.replace('"', '', regex=False)

In [5]:
y_train = df_train['Survived']
X_train = df_train.drop('Survived', axis=1)

In [6]:
X_test = df_test.copy()

In [7]:
# Check records from df_complete where `name` is not unique
df_complete[df_complete.duplicated(subset=['name'], keep=False)]

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
725,3,1,"Connolly, Miss. Kate",female,22.0,0,0,370373,7.7500,NaN,Q,13,NaN,Ireland
726,3,0,"Connolly, Miss. Kate",female,30.0,0,0,330972,7.6292,NaN,Q,NaN,NaN,Ireland
924,3,0,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q,NaN,70.0,NaN
925,3,0,"Kelly, Mr. James",male,44.0,0,0,363592,8.0500,NaN,S,NaN,NaN,NaN


In [8]:
# Merge the column `Survived` from df_complete as integer into df_test based on the `name` and `age` column from `df_complete` and `Name` and `Age` columns `df_test` ordered by `PassengerId`
y_test = df_test.merge(df_complete[['name', 'age', 'survived']], left_on=['Name', 'Age'], right_on=['name', 'age'], how='left')['survived']

In [9]:
# Display rows from `y_test` where `survived` is null
y_test

0      0
1      1
2      0
3      0
4      1
      ..
413    0
414    1
415    0
416    0
417    1
Name: survived, Length: 418, dtype: int64

In [10]:
from sap_rpt_oss import SAP_RPT_OSS_Classifier
from sklearn.metrics import accuracy_score

/home/user/projects/sap-tech-bytes/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Before running the next cell, make sure you are logged into the Hugging Face and are granted access to https://huggingface.co/SAP/sap-rpt-1-oss model there

`hf auth login --token $HF_TOKEN`

In [26]:
# Initialize a classifier
clf = SAP_RPT_OSS_Classifier(max_context_size=8192, bagging=8)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1182.09it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [41]:
!hf cache ls

ID                                           SIZE     LAST_ACCESSED  LAST_MODIFIED  REFS 
-------------------------------------------- -------- -------------- -------------- ---- 
model/SAP/sap-rpt-1-oss                         64.6M 13 minutes ago 14 minutes ago main 
model/sentence-transformers/all-MiniLM-L6-v2    91.6M 13 minutes ago 13 minutes ago main 

Found 2 repo(s) for a total of 2 revision(s) and 156.1M on disk.


In [40]:
!ls -l ~/.cache/huggingface/hub/

total 8
drwxr-sr-x 5 user group 4096 Feb 18 13:21 models--SAP--sap-rpt-1-oss
drwxr-sr-x 6 user group 4096 Feb 18 13:21 models--sentence-transformers--all-MiniLM-L6-v2


In [25]:
clf

,checkpoint,'2025-11-04_sap-rpt-one-oss.pt'
,bagging,8
,max_context_size,8192
,drop_constant_columns,True
,test_chunk_size,1000


In [43]:
clf.fit(X_train, y_train)

,checkpoint,'2025-11-04_sap-rpt-one-oss.pt'
,bagging,8
,max_context_size,8192
,drop_constant_columns,True
,test_chunk_size,1000


In [44]:
# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
# Predict labels
predictions = clf.predict(X_test)

Data type str not recognized! Defaulting to string
Data type str not recognized! Defaulting to string
Data type str not recognized! Defaulting to string
Data type str not recognized! Defaulting to string
Data type str not recognized! Defaulting to string
Data type str not recognized! Defaulting to string
Data type str not recognized! Defaulting to string
Data type str not recognized! Defaulting to string
Data type str not recognized! Defaulting to string
Data type str not recognized! Defaulting to string


In [45]:
print("Accuracy", accuracy_score(y_test, predictions))

Accuracy 0.7966507177033493


In [46]:
# Combine y_test, predictions into a dataframe
results_df = pd.DataFrame({'Actual': y_test, 'Predicted': predictions})

In [47]:
results_df.head(10)

,Actual,Predicted
0,0,0
1,1,0
2,0,0
3,0,0
4,1,1
5,1,0
6,0,1
7,1,0
8,1,1
9,0,0


In [48]:
grouped = results_df.groupby(['Actual', 'Predicted']).size()
print(grouped)

Actual  Predicted
0       0            229
        1             31
1       0             54
        1            104
dtype: int64
